# Notebook 02: Compliance Pattern

The CCA exam tests whether you understand that PCI compliance and data redaction must be **enforced programmatically in code**, not just requested in a system prompt. This notebook shows both patterns on the same scenario — a customer who includes a credit card number in their message — so you can see whether PII reaches the audit log.

Pattern covered: **PostToolUse compliance callback with regex redaction** vs. prompt-only compliance instructions.

## Setup

In [ ]:
import sys
from pathlib import Path

# Add project root so helpers and customer_service are importable
sys.path.insert(0, str(Path(".").resolve()))

import anthropic
from helpers import compare_results, print_usage

from customer_service.data.customers import CUSTOMERS
from customer_service.data.scenarios import SCENARIOS
from customer_service.services.audit_log import AuditLog
from customer_service.services.container import ServiceContainer
from customer_service.services.customer_db import CustomerDatabase
from customer_service.services.escalation_queue import EscalationQueue
from customer_service.services.financial_system import FinancialSystem
from customer_service.services.policy_engine import PolicyEngine

In [ ]:
def make_services() -> ServiceContainer:
    """Create a fresh ServiceContainer with seed customer data."""
    return ServiceContainer(
        customer_db=CustomerDatabase(CUSTOMERS),
        policy_engine=PolicyEngine(),
        financial_system=FinancialSystem(),
        escalation_queue=EscalationQueue(),
        audit_log=AuditLog(),
    )


from dotenv import find_dotenv, load_dotenv

# Load ANTHROPIC_API_KEY from .env (find_dotenv walks up from the notebooks/ dir)
load_dotenv(find_dotenv(), override=False)

client = anthropic.Anthropic()
scenario = SCENARIOS["happy_path"]  # C001, $50 refund
# Inject a credit card number into the message for compliance testing
pii_message = scenario["message"] + " My card is 4111-1111-1111-1111."
print(f"Customer ID: {scenario['customer_id']}")
print(f"Message with PII: {pii_message}")

## Anti-Pattern: Prompt-Only Compliance

<div style="border-left: 4px solid #dc3545; padding: 12px 16px; background: #fff5f5; margin: 8px 0;">
<strong>What's wrong:</strong> The system prompt says "never log credit card numbers" and gives Claude the exact redaction format, but nothing in code checks what actually reaches the audit log. Whether PII lands in the store depends entirely on what Claude chooses to write on a given call.
</div>

**What you will actually observe.** On a live run Claude almost always complies. The instruction is explicit, the format is spelled out, and a current model follows that kind of instruction nearly every time. So the cell below will usually show a redacted entry, or no card number at all, and the comparison with the correct pattern will look like a tie.

**Why that is still a failure.** "Almost always" is a rate, not a guarantee. A compliance audit does not accept a rate. And you cannot measure the rate from a notebook: a miss that happens one time in a few hundred calls will never show up here, but it will show up in production. Nothing in this design would tell you when it did. The deterministic replay section further down shows what the callback does on the call where Claude does write the raw number.


In [ ]:
from customer_service.anti_patterns.prompt_compliance import (
    run_prompt_compliance_agent,
)

anti_services = make_services()
print("Running anti-pattern (prompt-only compliance)...")
anti_result = run_prompt_compliance_agent(client, anti_services, pii_message)
print(f"Stop reason: {anti_result.stop_reason}")
print(f"Tool calls: {[tc['name'] for tc in anti_result.tool_calls]}")

In [ ]:
# Check whether the raw card number reached the audit log
# Read the store, not the tool result. Match any 16-digit form: dashes, spaces, dots, or none.
import re

RAW_CARD_PATTERN = re.compile(r"4111[-.\s]?1111[-.\s]?1111[-.\s]?1111")

anti_logs = anti_services.audit_log.get_entries()
print(f"Audit log entries: {len(anti_logs)}")

anti_pii_leaked = False
for entry in anti_logs:
    details = entry.details
    if RAW_CARD_PATTERN.search(details):
        anti_pii_leaked = True
        print(f"PII LEAKED in audit log: {details[:120]}")
    elif "****" in details:
        print(f"Claude redacted before logging: {details[:100]}")
    else:
        print(f"Claude left the card out of the log: {details[:100]}")

if not anti_pii_leaked:
    print("\nNo leak this run. That is the usual outcome, and it proves nothing about the next run.")


In [ ]:
class _UsageWrapper:
    def __init__(self, u):
        self.usage = u


print_usage(_UsageWrapper(anti_result.usage))

## Correct Pattern: Programmatic Compliance via Callback

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>Why this works:</strong> The PostToolUse compliance callback intercepts every <code>log_interaction</code> call and applies a regex to redact 16-digit card numbers before they reach the audit log. This is deterministic — the regex fires regardless of whether Claude remembered to redact, regardless of prompt wording, on every single call.
</div>

In [ ]:
from customer_service.agent import build_callbacks, get_system_prompt, run_agent_loop

correct_services = make_services()
callbacks = build_callbacks()
print("Running correct pattern (programmatic compliance callback)...")
correct_result = run_agent_loop(
    client,
    correct_services,
    pii_message,
    get_system_prompt(),
    callbacks=callbacks,
)
print(f"Stop reason: {correct_result.stop_reason}")
print(f"Tool calls: {[tc['name'] for tc in correct_result.tool_calls]}")

In [ ]:
# Verify that the raw card number never reached the audit log
# Same pattern as the anti-pattern check, so the comparison is like for like.
correct_logs = correct_services.audit_log.get_entries()
print(f"Audit log entries: {len(correct_logs)}")

correct_pii_safe = True
for entry in correct_logs:
    details = entry.details
    if RAW_CARD_PATTERN.search(details):
        correct_pii_safe = False
        print(f"PII LEAKED (unexpected): {details[:120]}")
    elif "****" in details:
        print(f"Redacted entry: {details[:100]}")
    else:
        print(f"Entry without card number: {details[:100]}")

print(f"\nPII safely redacted in audit log: {correct_pii_safe}")
if correct_pii_safe:
    print("No raw card number in the store. On a typical run this looks the same as the")
    print("anti-pattern, because Claude redacted before the callback ever saw the number.")


In [ ]:
print_usage(_UsageWrapper(correct_result.usage))

## Compare Results

In [ ]:
compare_results(
    {
        "pii_leaked": anti_pii_leaked,
        "audit_log_entries": len(anti_logs),
        "tool_calls": len(anti_result.tool_calls),
    },
    {
        "pii_leaked": not correct_pii_safe,
        "audit_log_entries": len(correct_logs),
        "tool_calls": len(correct_result.tool_calls),
    },
)

## Deterministic Replay: Same Log Call, Both Patterns

The live runs above usually agree with each other. The anti-pattern prompt gives Claude the exact redaction format, and a current model follows a clear instruction like that almost every time. So on a typical run the callback never sees a raw card number, and the comparison table shows no difference. That does not mean the callback adds nothing. It means a live model is the wrong instrument for showing a guarantee.

The callback's value is that it does not depend on Claude at all. To see it, hand `log_interaction` a raw card number directly with a scripted client, no API calls, and read the audit log store afterward. Same transcript, with and without callbacks. The transcript below is what a leaking run looks like: Claude looks up the customer, then logs the interaction with the card number still in the details.


In [ ]:
from helpers import scripted_client, text_turn, tool_turn

RAW_CARD = "4111-1111-1111-1111"


def leaking_transcript(card: str) -> list:
    """Claude looks up C001, then logs the interaction with the card number in the details."""
    return [
        tool_turn("lookup_customer", {"customer_id": "C001"}, "toolu_01"),
        tool_turn(
            "log_interaction",
            {
                "customer_id": "C001",
                "action": "refund_request",
                "details": f"Customer requested $50 refund. Card on file: {card}",
            },
            "toolu_02",
        ),
        text_turn("Your refund request has been logged."),
    ]


def replay_log(card: str, callbacks: dict | None) -> list[str]:
    """Replay the transcript and return what actually landed in the audit log store."""
    services = make_services()
    client = scripted_client(leaking_transcript(card), on_forced=text_turn("done"))
    run_agent_loop(client, services, pii_message, get_system_prompt(), callbacks=callbacks)
    return [entry.details for entry in services.audit_log.get_entries()]


def show(label: str, card: str, entries: list[str]) -> bool:
    """Print the audit log entries and return True if the raw card number is in the store."""
    leaked = any(card in e for e in entries)
    print(f"{label}")
    for e in entries:
        print(f"  audit log: {e}")
    print(f"  raw card in audit log: {leaked}\n")
    return leaked


In [ ]:
anti_leak = show("ANTI-PATTERN (no callbacks)", RAW_CARD, replay_log(RAW_CARD, callbacks=None))
correct_leak = show("CORRECT PATTERN (compliance callback)", RAW_CARD, replay_log(RAW_CARD, callbacks=build_callbacks()))

compare_results({"pii_safe": not anti_leak}, {"pii_safe": not correct_leak})
print("\nSame tool call. Without the callback the raw number is written to the audit log.")
print("With it, the regex runs before the write and only the last four digits survive.")
print("This holds on every run because nothing about it depends on what Claude chose to do.")


In [ ]:
# What the callback does NOT guarantee: the regex requires separators between the groups.
# A card number typed or normalized without dashes or spaces passes through untouched.
UNSEPARATED = "4111111111111111"

show("CORRECT PATTERN, card without separators", UNSEPARATED, replay_log(UNSEPARATED, callbacks=build_callbacks()))
print("A programmatic control is only as good as the pattern it checks. The guarantee is")
print("'this regex always runs', not 'no card number can ever reach the log'. When you rely on")
print("code for compliance, the code has to cover the inputs you will actually see.")


> **CCA Exam Tip:** If an exam answer says 'add PCI compliance rules to the system prompt,' it is WRONG. Compliance must be enforced programmatically through validation hooks and callbacks. System prompts provide context; code enforces rules. Prompt instructions are probabilistic guidance — a regex in a PostToolUse callback is a deterministic guarantee.

## Summary

- **Anti-pattern failure:** Prompt-only compliance is probabilistic. On live runs Claude almost always redacts, because the prompt spells out the format, so the live cells rarely show a leak. That is the problem, not the reassurance: a miss rate you cannot observe in a notebook is still a miss rate, and a single miss is a PCI violation with nothing in code to catch it.
- **Correct pattern guarantee:** The PostToolUse compliance callback runs a regex over the `log_interaction` details before the handler writes to the audit log, on every call, regardless of what Claude did. The deterministic replay shows this directly: the same raw-card tool call leaves the raw number in the store without the callback and `****-****-****-1111` with it.
- **What the guarantee covers:** Only what the regex matches. Dashed and spaced card numbers are redacted. An unseparated 16-digit number is not, as the last replay cell shows. Programmatic enforcement is only as strong as the pattern it checks.
- **Key principle:** Compliance enforcement belongs in code (callbacks), not in natural language (system prompts). This is CCA architectural principle #2: programmatic hooks are the only reliable enforcement mechanism.
- **Verification habit:** Judge every run by the audit log store, not by what Claude said it did.
